# Q17 — Qwen1.5-7B: Linear Probing

**Mirrors:** `17_[Analysis]_Linear_Probing_Dual_Curves.ipynb` (LLaMA)

**Available probes:**
1. **Task-identity probe** (5-class, n=270) — same as LLaMA ✓
2. **Refusal-behaviour probe** (OR vs TARGET, binary) — n_OR=22, n_TGT=~221 ✓ (imbalanced but feasible)
3. **Refusal-type probe** (OR vs RH) — **SKIPPED** (RH n=1, cannot do 5-fold CV)

**Key questions:**
- Does task identity become decodable early (before constellation peak at L5)?
- Does the behavior (refusal vs answer) probe rise later than task identity?
- Are the qualitative patterns similar to LLaMA despite different layer counts?

**LLaMA reference values (from NB17):**
- Task probe: peak=0.986 at L6; L1=0.970
- Behavior probe: peak=0.939 at L15; L1=0.817

**Qwen architecture:** 31 layers; constellation peak at L5.

**No GPU required.**

In [ ]:
EMBEDDINGS_DIR = './embeddings_qwen'
FIGURES_DIR    = './figures'
KNOWN_PEAK_L   = 5   # Qwen constellation peak

import os
os.makedirs(FIGURES_DIR, exist_ok=True)
print('Config set.')

In [ ]:
# COLAB ONLY
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
# !cp -a "/content/drive/MyDrive/embeddings/overalign_eval_qwen_1_5/." {EMBEDDINGS_DIR}/.
print('(Colab cell skipped)')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate, permutation_test_score
from sklearn.metrics import balanced_accuracy_score, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 13, 'font.family': 'serif',
    'axes.titlesize': 14, 'axes.titleweight': 'bold',
    'axes.labelsize': 13, 'legend.fontsize': 12,
    'xtick.labelsize': 12, 'ytick.labelsize': 12,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 2.2,
})

PAL = {
    'task':    '#2C3E50',
    'refusal': '#E74C3C',
    'type':    '#8E44AD',
    'ref':     '#7F8C8D',
    'chance':  '#BDC3C7',
}
BENIGN_TASKS = ['sentiment_analysis', 'translate', 'cryptanalysis', 'rag_qa']
BAL_ACC      = make_scorer(balanced_accuracy_score)
CV_SPLITS    = 5
CV           = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=42)
PCA_K_MAX    = 50

print(f'✓ Libraries loaded (CV={CV_SPLITS}-fold, PCA_K_MAX={PCA_K_MAX})')

In [ ]:
def to_np(d):
    return {
        k: (v.float().numpy().astype(np.float32) if isinstance(v, torch.Tensor)
            else np.array([e.float().numpy().astype(np.float32) for e in v]))
        for k, v in d.items()
    }

csv_files = sorted(f for f in os.listdir(EMBEDDINGS_DIR) if f.endswith('.csv'))
csv_df    = pd.read_csv(os.path.join(EMBEDDINGS_DIR, csv_files[-1]))
torch_path = csv_df['torch_path'].iloc[0]
if not os.path.isabs(torch_path):
    torch_path = os.path.join(EMBEDDINGS_DIR, os.path.basename(torch_path))
torch_data = torch.load(torch_path, map_location='cpu')

embeddings_np        = to_np(torch_data['embeddings'])
texts                = torch_data['texts']
text_type_labels     = np.array(torch_data['text_type_labels'])
intended_task_labels = np.array(torch_data['intended_task_labels'])
response_labels      = csv_df['llm_evaluation'].values
refusal_labels       = csv_df['refusal_class'].values

LAYER_NAMES = sorted(
    [k for k in embeddings_np if 'input_norm' in k],
    key=lambda x: int(x.split('layer_')[1].split('_')[0])
)
LAYER_NUMS = [int(ln.split('layer_')[1].split('_')[0]) for ln in LAYER_NAMES]

print(f'Loaded: {len(texts)} samples | {len(LAYER_NAMES)} layers')

In [ ]:
refusing_mask = (refusal_labels == 'direct_refusal') | (refusal_labels == 'indirect_refusal')
answered_mask = refusal_labels == 'direct_answer'
harmful_mask  = text_type_labels == 'harmful_instruction'
benign_mask   = np.isin(intended_task_labels, BENIGN_TASKS)

OVER_REFUSAL_MASK = benign_mask  & refusing_mask
REFUSED_HARMFUL   = harmful_mask & refusing_mask
TARGET_MASK = (
    ((response_labels == 'cautious') | (response_labels == 'not_harmful')) & answered_mask
)

BEHAV_MASK = OVER_REFUSAL_MASK | TARGET_MASK
TYPE_MASK  = OVER_REFUSAL_MASK | REFUSED_HARMFUL

print(f'OR: {OVER_REFUSAL_MASK.sum()}  |  RH: {REFUSED_HARMFUL.sum()}  |  TGT: {TARGET_MASK.sum()}')
print(f'Behavior probe: n={BEHAV_MASK.sum()} (OR={OVER_REFUSAL_MASK.sum()}, TGT={TARGET_MASK.sum()})')
print(f'Type probe:     n={TYPE_MASK.sum()}  (OR={OVER_REFUSAL_MASK.sum()}, RH={REFUSED_HARMFUL.sum()})')

if REFUSED_HARMFUL.sum() < CV_SPLITS:
    print(f'⚠ RH n={REFUSED_HARMFUL.sum()} < CV_SPLITS={CV_SPLITS} — refusal-type probe SKIPPED.')
    RUN_TYPE_PROBE = False
else:
    RUN_TYPE_PROBE = True

# Check behavior probe feasibility
if OVER_REFUSAL_MASK.sum() < CV_SPLITS:
    print(f'⚠ OR n={OVER_REFUSAL_MASK.sum()} < CV_SPLITS={CV_SPLITS} — behavior probe SKIPPED.')
    RUN_BEHAV_PROBE = False
else:
    RUN_BEHAV_PROBE = True
    print(f'✓ Behavior probe will run (n={BEHAV_MASK.sum()})')

In [ ]:
# ── Probe helper ──────────────────────────────────────────────────────────────
# Pipeline: PCA(k) → StandardScaler → L2-LogisticRegression
# k = min(PCA_K_MAX, floor(n * (K-1)/K) - 1) to guarantee k < n_train

def make_pipe(n_total):
    min_train = (n_total * (CV_SPLITS - 1)) // CV_SPLITS
    k = min(PCA_K_MAX, min_train - 1)
    return Pipeline([
        ('pca',    PCA(n_components=k, random_state=42)),
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced',
                                       solver='lbfgs', random_state=42, multi_class='auto')),
    ]), k

def probe_layer(X, y):
    """Returns (mean_balanced_acc, std_balanced_acc)."""
    pipe, _ = make_pipe(len(y))
    res = cross_validate(pipe, X, y, cv=CV, scoring=BAL_ACC, n_jobs=-1)
    return res['test_score'].mean(), res['test_score'].std()

# Report k values
for name, n in [('Task', 270), ('Behavior', BEHAV_MASK.sum())]:
    if n >= CV_SPLITS:
        _, k = make_pipe(n)
        print(f'  {name} probe: n={n} → k={k}')

# Integer labels
all_tasks_sorted = sorted(set(intended_task_labels))
task_to_int      = {t: i for i, t in enumerate(all_tasks_sorted)}
y_task           = np.array([task_to_int[t] for t in intended_task_labels])
y_behav          = OVER_REFUSAL_MASK[BEHAV_MASK].astype(int)  # 1=OR, 0=TGT

In [ ]:
# ── Run probes at every layer ─────────────────────────────────────────────────
print('Running probes across all layers...')
results = []

for lname, lnum in zip(LAYER_NAMES, LAYER_NUMS):
    emb  = embeddings_np[lname]
    row  = {'layer': lnum}

    # Task probe (always)
    mu, sd = probe_layer(emb, y_task)
    row['task_mu'] = mu;  row['task_sd'] = sd

    # Behavior probe
    if RUN_BEHAV_PROBE:
        mu, sd = probe_layer(emb[BEHAV_MASK], y_behav)
        row['behav_mu'] = mu;  row['behav_sd'] = sd
    else:
        row['behav_mu'] = np.nan;  row['behav_sd'] = np.nan

    # Type probe (skip if RH too small)
    if RUN_TYPE_PROBE:
        y_type = OVER_REFUSAL_MASK[TYPE_MASK].astype(int)
        mu, sd = probe_layer(emb[TYPE_MASK], y_type)
        row['type_mu'] = mu;  row['type_sd'] = sd
    else:
        row['type_mu'] = np.nan;  row['type_sd'] = np.nan

    results.append(row)
    print(f'  L{lnum:02d}  task={row["task_mu"]:.3f}  behav={row["behav_mu"]:.3f}')

df = pd.DataFrame(results)
print('\nAll probes done.')

In [ ]:
# ── Figure 1: Task vs Behavior probe (two-panel) ──────────────────────────────
# Mirrors: fig_nb17_probe_dual (LLaMA)

# LLaMA reference from NB17
LLAMA_TASK_L1  = 0.970;  LLAMA_TASK_PEAK  = 0.986
LLAMA_BEHAV_L1 = 0.817;  LLAMA_BEHAV_PEAK = 0.939

ARDITI_LAYER = 3   # global direction active (LLaMA ref; Qwen equivalent unclear)
CONST_PEAK   = KNOWN_PEAK_L
layers       = df['layer'].values

fig, ax = plt.subplots(figsize=(7.0, 3.8))

# Task probe
ax.plot(layers, df['task_mu'], 'o-', color=PAL['task'], lw=2.2, zorder=3,
        label=f'Task-identity probe (5-class, n={len(texts)})')
ax.fill_between(layers, df['task_mu'] - df['task_sd'], df['task_mu'] + df['task_sd'],
                color=PAL['task'], alpha=0.12)

# Behavior probe
if RUN_BEHAV_PROBE:
    ax.plot(layers, df['behav_mu'], 's--', color=PAL['refusal'], lw=2.2, zorder=3,
            label=f'Refusal-behaviour probe (OR vs TGT, n={BEHAV_MASK.sum()})')
    ax.fill_between(layers, df['behav_mu'] - df['behav_sd'], df['behav_mu'] + df['behav_sd'],
                    color=PAL['refusal'], alpha=0.12)

# LLaMA reference values (dotted)
ax.axhline(LLAMA_TASK_PEAK,  color=PAL['task'],    lw=1.0, ls=':',  alpha=0.5,
           label=f'LLaMA task peak ({LLAMA_TASK_PEAK:.3f})')
ax.axhline(LLAMA_BEHAV_PEAK, color=PAL['refusal'], lw=1.0, ls='--', alpha=0.5,
           label=f'LLaMA behav peak ({LLAMA_BEHAV_PEAK:.3f})')

# Chance levels
ax.axhline(1/5, color=PAL['chance'], lw=1.0, ls=':')
ax.axhline(0.5, color=PAL['chance'], lw=1.0, ls='--', alpha=0.7)
ax.text(30, 1/5 + 0.012, '20% (5-class chance)', fontsize=8, color=PAL['chance'], ha='right')
ax.text(30, 0.5  + 0.012, '50% (binary chance)',  fontsize=8, color=PAL['chance'], ha='right')

# Reference verticals
ax.axvline(CONST_PEAK, color=PAL['ref'], lw=1.4, ls='-.',
           label=f'Qwen constellation peak (L{CONST_PEAK})')

ax.set_xlabel('Layer'); ax.set_ylabel('Balanced accuracy (5-fold CV)')
ax.set_title('Qwen1.5-7B — Task identity vs Refusal behaviour probe', pad=8)
ax.set_xlim(-0.5, 30); ax.set_ylim(0.0, 1.10)
ax.legend(fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.18),
          ncol=3, frameon=True)

plt.tight_layout(pad=0.8)
plt.subplots_adjust(bottom=0.22)
plt.savefig(f'{FIGURES_DIR}/q_fig17_probe_dual.pdf', bbox_inches='tight', dpi=300)
plt.savefig(f'{FIGURES_DIR}/q_fig17_probe_dual.png', bbox_inches='tight', dpi=300)
plt.show()

print('=== FIGURE DATA (Probe Dual) ===')
print(f'  Qwen  task:  L1={df[df["layer"]==1]["task_mu"].values[0]:.3f}  peak={df["task_mu"].max():.3f}')
print(f'  LLaMA task:  L1={LLAMA_TASK_L1}  peak={LLAMA_TASK_PEAK}')
if RUN_BEHAV_PROBE:
    print(f'  Qwen  behav: L1={df[df["layer"]==1]["behav_mu"].values[0]:.3f}  peak={df["behav_mu"].max():.3f}')
    print(f'  LLaMA behav: L1={LLAMA_BEHAV_L1}  peak={LLAMA_BEHAV_PEAK}')

In [ ]:
# ── Figure 2: All-3-panel comparison (publication figure) ────────────────────
# Matches ACL fig_nb17_probe_all3 format
# Panel (a): task;  (b): behavior;  (c): type (shown but with note if skipped)

layers = df['layer'].values
fig, axes = plt.subplots(1, 3, figsize=(7.0, 3.8), sharey=False)
plt.subplots_adjust(wspace=0.35)

REF_KW_DASH = dict(color=PAL['ref'], lw=1.3, ls='-.', zorder=2)
CHANCE_KW   = dict(color=PAL['chance'], lw=1.0, zorder=1)

def add_peak(ax):
    ax.axvline(CONST_PEAK, **REF_KW_DASH)

# (a) Task probe
ax = axes[0]
ax.plot(layers, df['task_mu'], 'o-', color=PAL['task'], lw=2.2, zorder=3)
ax.fill_between(layers, df['task_mu']-df['task_sd'], df['task_mu']+df['task_sd'],
                color=PAL['task'], alpha=0.12)
ax.axhline(1/5, ls=':', **CHANCE_KW)
ax.text(30, 1/5 + 0.015, '20%', fontsize=8, color=PAL['chance'], ha='right')
add_peak(ax)
ax.text(CONST_PEAK + 0.4, 1.04, f'L{CONST_PEAK}', fontsize=8, color=PAL['ref'], va='top')
ax.set_xlabel('Layer'); ax.set_ylabel('Balanced accuracy')
ax.set_title(f'(a) Task identity\n(5-class, n={len(texts)})', pad=6, fontsize=12)
ax.set_xlim(-0.5, 30); ax.set_ylim(0.0, 1.12)
ax.text(12, 0.50, 'Task probe', fontsize=9, color=PAL['task'], fontweight='bold')

# (b) Refusal-behaviour probe
ax = axes[1]
if RUN_BEHAV_PROBE:
    ax.plot(layers, df['behav_mu'], 's--', color=PAL['refusal'], lw=2.2, zorder=3)
    ax.fill_between(layers, df['behav_mu']-df['behav_sd'], df['behav_mu']+df['behav_sd'],
                    color=PAL['refusal'], alpha=0.12)
    ax.text(12, 0.56, 'Refusal-behav probe', fontsize=9, color=PAL['refusal'], fontweight='bold')
else:
    ax.text(0.5, 0.5, 'OR n too small\nfor CV', transform=ax.transAxes,
            ha='center', va='center', color='red', fontsize=11)
ax.axhline(0.5, ls='--', **CHANCE_KW)
ax.text(30, 0.5 + 0.015, '50%', fontsize=8, color=PAL['chance'], ha='right')
add_peak(ax)
ax.set_xlabel('Layer')
ax.set_title(f'(b) Refusal behaviour\n(OR vs TGT, n={BEHAV_MASK.sum()})', pad=6, fontsize=12)
ax.set_xlim(-0.5, 30); ax.set_ylim(0.4, 1.12)

# (c) Refusal-type probe
ax = axes[2]
if RUN_TYPE_PROBE:
    ax.plot(layers, df['type_mu'], '^-.', color=PAL['type'], lw=2.2, zorder=3)
    ax.fill_between(layers, df['type_mu']-df['type_sd'], df['type_mu']+df['type_sd'],
                    color=PAL['type'], alpha=0.12)
    ax.text(12, 0.56, 'Refusal-type probe', fontsize=9, color=PAL['type'], fontweight='bold')
else:
    ax.text(0.5, 0.5,
            f'⚠ Skipped\nRH n={REFUSED_HARMFUL.sum()} < {CV_SPLITS}\n(Qwen near-zero refusal rate)',
            transform=ax.transAxes, ha='center', va='center', color='#8E44AD', fontsize=10)
    ax.text(0.5, 0.15,
            'LLaMA: 100% from L1\n(for reference)',
            transform=ax.transAxes, ha='center', va='bottom', color=PAL['ref'], fontsize=9)
ax.axhline(0.5, ls='--', **CHANCE_KW)
ax.text(30, 0.5 + 0.015, '50%', fontsize=8, color=PAL['chance'], ha='right')
add_peak(ax)
ax.set_xlabel('Layer')
ax.set_title(f'(c) Refusal type\n(OR vs RH — n={TYPE_MASK.sum()})', pad=6, fontsize=12)
ax.set_xlim(-0.5, 30); ax.set_ylim(0.3, 1.12)

# Legend
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0],[0], color=PAL['ref'], lw=1.3, ls='-.', label=f'L{CONST_PEAK} — Qwen constellation peak'),
]
fig.legend(handles=legend_handles, fontsize=9,
           loc='lower center', bbox_to_anchor=(0.5, -0.06), ncol=2, frameon=True)

plt.suptitle('Qwen1.5-7B — Linear probes: task identity and refusal behaviour\n'
             '(Refusal-type probe skipped: Qwen harmful-refusal rate ≈ 0%)',
             fontsize=10, fontweight='bold', y=1.03)
plt.tight_layout(pad=0.6)
plt.subplots_adjust(bottom=0.14)
plt.savefig(f'{FIGURES_DIR}/q_fig17_probe_all3.pdf', bbox_inches='tight', dpi=300)
plt.savefig(f'{FIGURES_DIR}/q_fig17_probe_all3.png', bbox_inches='tight', dpi=300)
plt.show()
print('✓ Saved: q_fig17_probe_all3')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('=' * 65)
print('Q17 — QWEN LINEAR PROBING SUMMARY')
print('=' * 65)

print(f'\nTask probe (n=270, 5-class):')
print(f'  Qwen  — L1: {df[df["layer"]==1]["task_mu"].values[0]:.3f}  '
      f'peak: {df["task_mu"].max():.3f} at L{df["layer"][df["task_mu"].argmax()]}')
print(f'  LLaMA — L1: 0.970  peak: 0.986 at L6')

if RUN_BEHAV_PROBE:
    print(f'\nBehavior probe (n={BEHAV_MASK.sum()}, OR={OVER_REFUSAL_MASK.sum()} vs TGT={TARGET_MASK.sum()}):')
    print(f'  Qwen  — L1: {df[df["layer"]==1]["behav_mu"].values[0]:.3f}  '
          f'peak: {df["behav_mu"].max():.3f} at L{df["layer"][df["behav_mu"].argmax()]}')
    print(f'  LLaMA — L1: 0.817  peak: 0.939 at L15')

print(f'\nRefusal-type probe: {"SKIPPED" if not RUN_TYPE_PROBE else "ran"}')
if not RUN_TYPE_PROBE:
    print(f'  Reason: RH n={REFUSED_HARMFUL.sum()} < CV_SPLITS={CV_SPLITS}')
    print(f'  LLaMA reference: 100% balanced accuracy from L1 onward')
print()
print('KEY OBSERVATION:')
print('  If task probe rises before behavior probe, the pattern mirrors LLaMA:')
print('  task identity is encoded before the refusal decision is made.')
print('  This is consistent with over-refusal being task-conditioned in Qwen too.')
print('=' * 65)